# 🌽 Training YOLOv8 - Deteksi Penyakit Daun Jagung
### Notebook ini akan menghasilkan:
- ✅ Grafik Training (Loss, Precision, Recall, mAP)
- ✅ Tabel Evaluasi per Kelas (Precision, Recall, F1, mAP50)
- ✅ Confusion Matrix
- ✅ Model best.pt siap pakai

> ⚡ **Aktifkan GPU dulu:** Runtime → Change runtime type → **T4 GPU** → Save

## ✅ Step 1: Cek GPU

In [ ]:
!nvidia-smi
import torch
print(f'\n🔥 CUDA: {torch.cuda.is_available()}')
print(f'🖥️  GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tidak terdeteksi"}')

## ✅ Step 2: Install Library

In [ ]:
!pip install ultralytics -q
print('✅ Ultralytics berhasil diinstall!')

## ✅ Step 3: Upload Dataset ZIP
> Upload file **`Jagung.zip`** dari folder `Jagung_Deteksi Percobaan`

In [ ]:
from google.colab import files
import zipfile, os

print('📂 Silakan upload file ZIP dataset...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\n📦 Mengekstrak: {zip_name}')

os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print('\n✅ Dataset berhasil diekstrak!')
print('\nIsi folder dataset:')
for item in os.listdir('/content/dataset'):
    print(f'  - {item}')

## ✅ Step 4: Setup Path data.yaml

In [ ]:
import glob, yaml, os

yaml_files = glob.glob('/content/dataset/**/*.yaml', recursive=True)
if not yaml_files:
    yaml_files = glob.glob('/content/dataset/*.yaml')

yaml_path = yaml_files[0]
print(f'✅ data.yaml ditemukan: {yaml_path}')

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

dataset_root = os.path.dirname(yaml_path)

data['train'] = os.path.join(dataset_root, 'train', 'images')
data['val']   = os.path.join(dataset_root, 'valid', 'images')
data['test']  = os.path.join(dataset_root, 'test',  'images')

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('\n📋 Konfigurasi dataset:')
print(f'   train : {data["train"]}')
print(f'   val   : {data["val"]}')
print(f'   test  : {data["test"]}')
print(f'   nc    : {data.get("nc", "?")} kelas')
print(f'   names : {data.get("names", "?")}')

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_root, split, 'images')
    if os.path.exists(img_dir):
        count = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f'   {split:6s} images: {count}')

## ✅ Step 5: Training Model YOLOv8
> **Epoch: 40** | **Image Size: 640** | **Batch: 16**

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

print('🚀 Memulai training...')

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project='/content/runs',
    name='jagung',
    exist_ok=True,
    plots=True,
    save=True,
)

print('\n✅ Training selesai!')

## ✅ Step 6: Tampilkan Grafik Training

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

run_dir = '/content/runs/jagung'
results_img = os.path.join(run_dir, 'results.png')

if os.path.exists(results_img):
    img = mpimg.imread(results_img)
    plt.figure(figsize=(18, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Grafik Training YOLOv8 - Penyakit Daun Jagung', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/grafik_training.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('✅ Grafik training ditampilkan!')
else:
    print('❌ results.png tidak ditemukan')

## ✅ Step 7: Tampilkan Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

run_dir = '/content/runs/jagung'

cm_files = glob.glob(os.path.join(run_dir, 'confusion_matrix_normalized.png'))
if not cm_files:
    cm_files = glob.glob(os.path.join(run_dir, 'confusion_matrix.png'))

if cm_files:
    img = mpimg.imread(cm_files[0])
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix Normalized', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('✅ Confusion matrix ditampilkan!')
else:
    print('❌ confusion_matrix.png tidak ditemukan')

## ✅ Step 8: Evaluasi pada Test Set - Tabel Metrik per Kelas

In [ ]:
from ultralytics import YOLO
import numpy as np

best_model = YOLO('/content/runs/jagung/weights/best.pt')

print('🔍 Mengevaluasi pada TEST SET...')
metrics = best_model.val(
    data=yaml_path,
    split='test',
    verbose=False
)

class_names = data.get('names', [])
precision   = metrics.box.p
recall      = metrics.box.r
f1          = metrics.box.f1
map50       = metrics.box.ap50

class_name_map = {
    'blight'        : 'Blight',
    'common_rust'   : 'Common Rust',
    'gray_leaf_spot': 'Gray Leaf Spot',
    'healthy'       : 'Healthy',
}

mp   = metrics.box.mp
mr   = metrics.box.mr
mf1  = float(np.mean(f1)) if f1 is not None and len(f1) > 0 else 0
mmap = metrics.box.map50

print('\n' + '='*70)
print('       TABEL 4.4 HASIL EVALUASI MODEL PER KELAS (TEST SET)')
print('='*70)
print(f'{"Kelas":<20} {"Precision":>12} {"Recall":>10} {"F1-Score":>10} {"mAP50":>10}')
print('-'*70)

for i, cls in enumerate(class_names):
    display_name = class_name_map.get(cls, cls)
    p = precision[i] if i < len(precision) else 0
    r = recall[i]    if i < len(recall)    else 0
    f = f1[i]        if i < len(f1)        else 0
    m = map50[i]     if i < len(map50)     else 0
    print(f'{display_name:<20} {p:>12.4f} {r:>10.4f} {f:>10.4f} {m:>10.4f}')

print('-'*70)
print(f'{"Rata-rata":<20} {mp:>12.4f} {mr:>10.4f} {mf1:>10.4f} {mmap:>10.4f}')
print('='*70)

## ✅ Step 9: Simpan Tabel Metrik sebagai Gambar

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rows = []
for i, cls in enumerate(class_names):
    display_name = class_name_map.get(cls, cls)
    p = precision[i] if i < len(precision) else 0
    r = recall[i]    if i < len(recall)    else 0
    f = f1[i]        if i < len(f1)        else 0
    m = map50[i]     if i < len(map50)     else 0
    rows.append([display_name,
                 f'{p:.4f}'.replace('.', ','),
                 f'{r:.4f}'.replace('.', ','),
                 f'{f:.4f}'.replace('.', ','),
                 f'{m:.4f}'.replace('.', ',')])

rows.append(['Rata-rata',
             f'{mp:.4f}'.replace('.', ','),
             f'{mr:.4f}'.replace('.', ','),
             f'{mf1:.4f}'.replace('.', ','),
             f'{mmap:.4f}'.replace('.', ',')])

col_labels = ['Kelas', 'Precision', 'Recall', 'F1-Score', 'mAP50']

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis('off')

table = ax.table(
    cellText=rows,
    colLabels=col_labels,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.0)

for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')

last_row = len(rows)
for j in range(len(col_labels)):
    table[last_row, j].set_facecolor('#D5DBDB')
    table[last_row, j].set_text_props(fontweight='bold')

for i in range(1, len(rows)):
    for j in range(len(col_labels)):
        if i % 2 == 0:
            table[i, j].set_facecolor('#EBF5FB')

plt.title('Tabel 4.4 Hasil Evaluasi Model per Kelas', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/content/tabel_evaluasi.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Tabel evaluasi disimpan!')

## ✅ Step 10: Download Semua Hasil (ZIP)

In [ ]:
import shutil, os
from google.colab import files

os.makedirs('/content/hasil_training', exist_ok=True)

run_dir = '/content/runs/jagung'

files_to_copy = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
    'labels.jpg',
    'results.csv',
]

print('📦 Mengemas file hasil...')
for fname in files_to_copy:
    src = os.path.join(run_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, '/content/hasil_training/')
        print(f'  ✅ {fname}')
    else:
        print(f'  ⚠️  {fname} tidak ada')

for extra in ['/content/grafik_training.png', '/content/tabel_evaluasi.png', '/content/confusion_matrix.png']:
    if os.path.exists(extra):
        shutil.copy(extra, '/content/hasil_training/')

weights_dir = os.path.join(run_dir, 'weights')
if os.path.exists(weights_dir):
    shutil.copytree(weights_dir, '/content/hasil_training/weights', dirs_exist_ok=True)
    print('  ✅ weights/ (best.pt & last.pt)')

shutil.make_archive('/content/hasil_training_jagung', 'zip', '/content/hasil_training')
print('\n📥 Mendownload ZIP...')
files.download('/content/hasil_training_jagung.zip')
print('✅ Download dimulai! Cek folder Downloads Anda.')